In [14]:
import psycopg2
import csv
from psycopg2.extras import execute_values
from datetime import datetime  

# Aqui van las rutas de los archivos TXT
ruta_productos_txt = 'C:/Users\MATI_/OneDrive\Escritorio/Assessment_Data_Engineer/Archivos_verificados/productos_correctos.txt'
ruta_tickets_txt = 'C:/Users\MATI_/OneDrive\Escritorio/Assessment_Data_Engineer/Archivos_verificados/tickets_correctos.txt'

def leer_datos_txt(ruta_archivo):
    """Aqui se lee datos desde un archivo TXT, limpiando valores y manejando None, sin verificación de longitud."""
    try:
        with open(ruta_archivo, 'r', encoding='utf-8', errors='ignore') as archivo:
            lector_csv = csv.reader(archivo, delimiter=';')
            encabezados = next(lector_csv)
            datos = []

            for fila in lector_csv:
                # Aqui se realiza la verificación de longitud se ha eliminado
                fila_dict = dict(zip(encabezados, fila))

                # Aqui se realiza la limpieza y manejo de valores faltantes (sin cambios):
                for clave, valor in fila_dict.items():
                    if valor:
                        valor = valor.strip().replace('\r', '').replace('\n', '').replace('\t', '')
                        if valor == "" or valor is None or valor == "None":
                            fila_dict[clave] = None
                        else:
                            fila_dict[clave] = valor
                    else:
                        fila_dict[clave] = None

                datos.append(fila_dict)
            return datos

    except FileNotFoundError:
        print(f"Archivo no encontrado: {ruta_archivo}")
        return None
    except Exception as e:
        print(f"Error inesperado al leer {ruta_archivo}: {e}")
        return None


def convertir_a_entero(valor):
    """Aqui se convierte un valor a entero, manejando None y cadenas vacías."""
    if valor and not isinstance(valor, datetime):  # Aqui se verifica que exista y no sea datetime
        try:
            return int(valor)
        except ValueError:
            print(f"Error de conversión a entero: {valor}")
            return None
    return None

def normalizar_texto(texto):
    """Aqui se normaliza un texto a minúsculas y elimina espacios en blanco."""
    if texto:
        return texto.lower().strip()
    return ""

def normalizar_datos_productos(datos_productos):
    """Aqui se normaliza los datos de productos."""
    datos_normalizados = []
    for producto in datos_productos:
        producto_normalizado = {}

        # Aqui se realizan las conversiones y normalizaciones de productos, manejando None:

        producto_normalizado['idcadena'] = convertir_a_entero(producto.get('idcadena'))
        producto_normalizado['eancode'] = normalizar_texto(producto.get('eancode'))
        producto_normalizado['descripcion'] = normalizar_texto(producto.get('descripcion'))
        producto_normalizado['id_sector'] = convertir_a_entero(producto.get('id_sector'))
        producto_normalizado['sector'] = normalizar_texto(producto.get('sector'))
        producto_normalizado['id_seccion'] = convertir_a_entero(producto.get('id_seccion'))
        producto_normalizado['seccion'] = normalizar_texto(producto.get('seccion'))
        producto_normalizado['id_categoria'] = convertir_a_entero(producto.get('id_categoria'))
        producto_normalizado['categoria'] = normalizar_texto(producto.get('categoria'))
        producto_normalizado['id_subcategoria'] = convertir_a_entero(producto.get('id_subcategoria'))
        producto_normalizado['subcategoria'] = normalizar_texto(producto.get('subcategoria'))
        producto_normalizado['fabricante'] = normalizar_texto(producto.get('fabricante'))
        producto_normalizado['marca'] = normalizar_texto(producto.get('marca'))
        producto_normalizado['contenido'] = normalizar_texto(producto.get('contenido'))

        pesovolumen_str = producto.get('pesovolumen')
        producto_normalizado['pesovolumen'] = float(pesovolumen_str) if pesovolumen_str and pesovolumen_str != 'None' else None

        producto_normalizado['unidadmedida'] = normalizar_texto(producto.get('unidadmedida'))

        ultmodificacion_str = producto.get('ultmodificacion')
        try:
            producto_normalizado['ultmodificacion'] = datetime.strptime(ultmodificacion_str, '%Y-%m-%d %H:%M:%S.%f') if ultmodificacion_str else None
        except ValueError:
            print(f"Error de formato de fecha: {ultmodificacion_str}")
            producto_normalizado['ultmodificacion'] = None

        producto_normalizado['id'] = convertir_a_entero(producto.get('id'))
        producto_normalizado['granfamilia'] = normalizar_texto(producto.get('granfamilia'))
        producto_normalizado['familia'] = normalizar_texto(producto.get('familia'))
        producto_normalizado['categoria_nueva'] = normalizar_texto(producto.get('categoria_nueva'))
        producto_normalizado['subcategoria_nueva'] = normalizar_texto(producto.get('subcategoria_nueva'))

        datos_normalizados.append(producto_normalizado)
    return datos_normalizados

def normalizar_datos_tickets(datos_tickets):
    """
    Aqui se normalizan datos de tickets, manejando todas las columnas.
    """
    datos_normalizados = []
    for ticket in datos_tickets:
        ticket_normalizado = {}

         # Aqui se realizan las conversiones y normalizaciones de tickets, manejando None:
        ticket_normalizado['punto'] = convertir_a_entero(ticket.get('punto'))
        ticket_normalizado['ticket'] = normalizar_texto(ticket.get('ticket'))

        fecha_str = ticket.get('fecha')
        try:
            ticket_normalizado['fecha'] = datetime.strptime(fecha_str, '%Y-%m-%d').date() if fecha_str else None
        except ValueError:
            print(f"Error de formato de fecha: {fecha_str}")
            ticket_normalizado['fecha'] = None

        hora_str = ticket.get('hora')
        try:
            ticket_normalizado['hora'] = datetime.strptime(hora_str, '%H:%M:%S').time() if hora_str else None
        except ValueError:
            print(f"Error de formato de hora: {hora_str}")
            ticket_normalizado['hora'] = None

        ticket_normalizado['eancode'] = normalizar_texto(ticket.get('eancode'))
        ticket_normalizado['ean_desc'] = normalizar_texto(ticket.get('ean_desc'))

        unidades_vendidas_str = ticket.get('unidades_vendidas')
        ticket_normalizado['unidades_vendidas'] = float(unidades_vendidas_str) if unidades_vendidas_str and unidades_vendidas_str != 'None' else None

        precio_regular_str = ticket.get('precio_regular')
        ticket_normalizado['precio_regular'] = float(precio_regular_str) if precio_regular_str and precio_regular_str != 'None' else None

        precio_promocional_str = ticket.get('precio_promocional')
        ticket_normalizado['precio_promocional'] = float(precio_promocional_str) if precio_promocional_str and precio_promocional_str != 'None' else None

        ticket_normalizado['tipo_venta'] = normalizar_texto(ticket.get('tipo_venta'))
        ticket_normalizado['idcadena'] = convertir_a_entero(ticket.get('idcadena'))

        ultmodificacion_str = ticket.get('ultmodificacion')
        try:
            ticket_normalizado['ultmodificacion'] = datetime.strptime(ultmodificacion_str, '%Y-%m-%d %H:%M:%S.%f') if ultmodificacion_str else None
        except ValueError:
            print(f"Error de formato de fecha/hora: {ultmodificacion_str}")
            ticket_normalizado['ultmodificacion'] = None

        anulado_str = ticket.get('anulado')
        ticket_normalizado['anulado'] = bool(anulado_str) if anulado_str else None

        ticket_normalizado['id'] = convertir_a_entero(ticket.get('id'))

        datos_normalizados.append(ticket_normalizado)
    return datos_normalizados

# Aqui se leen los datos desde los archivos TXT
datos_productos_leidos = leer_datos_txt(ruta_productos_txt)
datos_tickets_leidos = leer_datos_txt(ruta_tickets_txt)
    
# Aqui se ejecuta la función y guarda el resultado
datos_productos_normalizados = normalizar_datos_productos(datos_productos_leidos) 
datos_tickets_normalizados = normalizar_datos_tickets(datos_tickets_leidos) 

# Aqui se realiza la conexión a la BD
conexion = psycopg2.connect(
    host="127.0.0.1",
    database="Ventas_Productos",
    user="admin",
    password="1234",
    port=1024
)

def cargar_datos(datos, nombre_tabla, conexion):
    """
    Carga datos en una tabla PostgreSQL, adaptada a las últimas normalizaciones.
    """
    cursor = conexion.cursor()

    try:
        cursor.execute(f"SELECT column_name FROM information_schema.columns WHERE table_name = '{nombre_tabla}'")
        columnas = [row[0] for row in cursor.fetchall()]
        
        valores_a_insertar = [[fila.get(col, None) for col in columnas] for fila in datos]
        
        consulta = f"INSERT INTO {nombre_tabla} ({', '.join(columnas)}) VALUES %s"
        
        execute_values(cursor, consulta, valores_a_insertar)
        
        conexion.commit()
        print(f"Datos cargados en {nombre_tabla} exitosamente.")
    except Exception as e:
        conexion.rollback()
        print(f"Error en {nombre_tabla}: {e}")
    finally:
        cursor.close()

# Aqui se cargan los datos normalizados
cargar_datos(datos_productos_normalizados, 'productos', conexion)
cargar_datos(datos_tickets_normalizados, 'tickets', conexion)  

# Aqui se cierra la conexión
conexion.close()

Datos cargados en productos exitosamente.
Datos cargados en tickets exitosamente.


In [ ]:
import psycopg2

try:
    # Parámetros de conexión más detallados (opcional)
    conexion = psycopg2.connect(
        host="127.0.0.1",
        database="Ventas_Productos",
        user="admin",
        password="1234",
        port=1024  # Especifica el puerto si es necesario
    )

    # Si la conexión es exitosa, puedes ejecutar consultas aquí:
    with conexion.cursor() as cursor:
        cursor.execute("SELECT * FROM calendario")
        resultados = cursor.fetchall()
        print(resultados)

except psycopg2.OperationalError as e:
    print("Error al conectar a la base de datos:", e)
except Exception as e:
    print("Error inesperado:", e)

finally:
    if conexion:
        conexion.close()